# Занятие 2, демо 4. Примеры и вопросы

Все расчёты - на маленьких числах, чтобы каждый можно было проверить в уме.

## Пример 1. Затухание на трёх токенах

Ключи и запросы равны 1, значения 2, 4, 8, $\gamma=0.5$.

In [ ]:
values = [2.0, 4.0, 8.0]
gamma = 0.5

state = 0.0
for step, v in enumerate(values, 1):
    state = gamma * state + v
    print(f"после токена {step}: {state}")

weights = [gamma ** 2, gamma, 1.0]
print("сразу, с весами", weights, "->", sum(w * v for w, v in zip(weights, values)))

In [ ]:
state = values[0]
for v, g in zip(values[1:], [0.01, 0.5]):
    state = g * state + v
    print(f"гейт {g}: в памяти {state:.2f}")

print("от первого значения осталось:", values[0] * 0.01 * 0.5)

## Пример 2. Три способа посчитать одно и то же

In [ ]:
state = 0.0
for v in values:
    state = gamma * state + v
step_by_step = state

parallel = sum(gamma ** (len(values) - 1 - i) * v for i, v in enumerate(values))

after_first_chunk = gamma * values[0] + values[1]
chunkwise = gamma * after_first_chunk + values[2]

print("шаг за шагом:", step_by_step)
print("параллельно: ", parallel)
print("по кускам:   ", chunkwise)

## Пример 3. Коллизия при delta rule

Память из примера 2×2: $k_1=(1,0)$, $k_2=(0.8,0.6)$. Пишем по $k_1$ новое
значение $(2,0)$ при $\beta=1$.

In [ ]:
import torch

S = torch.tensor([[1.0, 0.0], [0.8, 0.6]], dtype=torch.float64)
k1 = torch.tensor([1.0, 0.0], dtype=torch.float64)
k2 = torch.tensor([0.8, 0.6], dtype=torch.float64)
v_new = torch.tensor([2.0, 0.0], dtype=torch.float64)

error = v_new - S @ k1
S_after = S + torch.outer(error, k1)

print("по k1 до правки:   ", (S @ k1).tolist())
print("ошибка:            ", error.tolist())
print("память после правки:", S_after.tolist())
print("по k1 после правки:", (S_after @ k1).tolist())

In [ ]:
print("по k2 до правки:   ", [round(x, 2) for x in (S @ k2).tolist()])
print("по k2 после правки:", [round(x, 2) for x in (S_after @ k2).tolist()])

In [ ]:
S_exact = torch.tensor([[2.0, -8 / 3], [0.0, 5 / 3]], dtype=torch.float64)
v2 = torch.tensor([0.0, 1.0], dtype=torch.float64)
print("точное решение по k1:", [round(x, 6) for x in (S_exact @ k1).tolist()])
print("точное решение по k2:", [round(x, 6) for x in (S_exact @ k2).tolist()])
print()

state = S_after.clone()
for lap in range(1, 7):
    state = state + torch.outer(v2 - state @ k2, k2)
    state = state + torch.outer(v_new - state @ k1, k1)
    print(f"круг {lap}: по k2 читается {[round(x, 3) for x in (state @ k2).tolist()]}")

## Пример 4. Две ручки Gated DeltaNet

Память из двух чисел: первое читается текущим ключом $k=(1,0)$, второе -
посторонняя старая информация. Пишем по ключу $v=2$.

In [ ]:
def gated_delta(S, k, v, alpha, beta):
    S = alpha * S
    return S + beta * (v - S @ k) * k


S_old = torch.tensor([10.0, 5.0], dtype=torch.float64)
k = torch.tensor([1.0, 0.0], dtype=torch.float64)

print("альфа 0.1, бета 1:", gated_delta(S_old, k, 2.0, alpha=0.1, beta=1.0).tolist())

In [ ]:
print("альфа 1, бета 1:  ", gated_delta(S_old, k, 2.0, alpha=1.0, beta=1.0).tolist())
print("затухание 0.1 и обычная запись:", (0.1 * S_old + 2.0 * k).tolist())

## Пример 5. Граница документов

Всё, что было до границы, больше не нужно. Как с этим справится каждый режим:
обычная память, затухание, delta rule, Gated DeltaNet? Какой ручкой?

## Пример 6. Иголка и много вопросов

Модель заявляет окно 1M токенов и показывает 99 % на иголке по всей длине.
Пройдёт ли она многозапросный тест на той же длине?

## Пример 7. Задача на проектирование

Модель с памятью фиксированного размера, заявленное окно 1M токенов. Как
проверить, пользуется ли она началом текста, и понять, где кончается её
память? Что меняете, что меряете, что держите неизменным? Что проверяет ваш
эксперимент - извлечение издалека или ёмкость?

## Вопросы для самопроверки

1. Что лежит в памяти с постоянным затуханием, если развернуть все шаги?
2. Что будет с тремя формами RetNet, если $\gamma=1$?
3. Почему при ключе единичной длины и $\beta=1$ правка точная?
4. Почему delta rule - это не нормализованное линейное внимание?
5. Когда запись по правилу дельты не меняет чтение по другому ключу?
6. Чем $\alpha$ отличается от $\beta$?
7. Почему одной $\alpha$ недостаточно для точной правки?
8. Почему одной $\beta$ недостаточно для общего забывания?
9. Что именно решает память фиксированного размера в задаче длинного контекста?

## Ещё вопросы

10. Почему почти перпендикулярные ключи не гарантируют, что помех нет?
11. Почему $\gamma$, зависящая от входа, - это ещё не Mamba?
12. Что будет в Gated DeltaNet при $\beta=0$?
13. Почему RetNet, Mamba и RWKV нельзя назвать одной архитектурой?
14. Почему по словам «длинная свёртка» нельзя судить о стоимости генерации?